<div style="border-top:4px solid #0f766e;padding:16px">DATA WAREHOUSING WITH APACHE DORIS · OPTIONAL LAB</div>

# 选做 Lab 5A：Kafka → Routine Load → Doris

目标：持续消费订单 JSON，查看进度，暂停后积压消息，再恢复并验证结果。
约 25–40 分钟，不影响 Level 1 主线完成。

先阅读[环境说明](../../environments/streaming/README.md)。需要 Docker Compose、网络下载权限及课程 Python 环境；仅用于本地测试。
下面会启动课程 Doris 和一个 Kafka 容器，并重建独立库 `dw_course_l1_streaming` 中的 `ext_kafka_orders`。
不修改主线订单表。每次使用新 Topic / Job；同一沙箱一次只运行一个本实验。

本例的输入是模拟业务订单，不是 MySQL Binlog；单分区按顺序发送，不验证跨分区乱序。

In [ ]:
from pathlib import Path
import os, sys
from uuid import uuid4
course = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dw_course").is_dir())
sys.path.insert(0, str(course))
from dw_course.docker_runtime import prepare_environment, connect_sandbox
from dw_course.runtime import expect
from dw_course.streaming import (
    prepare_streaming, kafka, produce, mysql, flink_api, submit_sql,
    wait_checkpoint, stop_with_savepoint, wait_rows,
)
prepare_environment(start=True)
os.environ["DW_DATABASE"] = "dw_course_l1_streaming"
lab = connect_sandbox()
print("独立实验库：", lab.database)

prepare_streaming("kafka", start=True)

## 1. 创建独立目标表、Topic 和常驻任务

先检查旧任务：若上次中断后仍在运行，按环境说明停止旧任务再重跑，不能在消费中重建表。
Routine Load 消费进度由 Doris 管理，并不是 Kafka 普通消费组提交的 Offset。

In [ ]:
expect(lab.query("SHOW ROUTINE LOAD"), [])
lab.execute("DROP TABLE IF EXISTS ext_kafka_orders")
lab.execute("""CREATE TABLE ext_kafka_orders (
    order_id BIGINT NOT NULL, customer_id BIGINT NOT NULL,
    order_status VARCHAR(32) NOT NULL, order_amount DECIMAL(18,2) NOT NULL
) UNIQUE KEY(order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES("replication_num"="1", "enable_unique_key_merge_on_write"="true")""")
run_id = uuid4().hex[:12]
topic = "course_orders_" + run_id
job_name = "course_orders_" + run_id
print(kafka("kafka-topics.sh", "--create", "--topic", topic, "--partitions", "1", "--replication-factor", "1"))
load_sql = f'''CREATE ROUTINE LOAD {job_name} ON ext_kafka_orders
COLUMNS(order_id, customer_id, order_status, order_amount)
PROPERTIES("format"="json", "desired_concurrent_number"="1",
           "max_batch_interval"="5", "max_error_number"="0", "strict_mode"="true")
FROM KAFKA("kafka_broker_list"="course-stream-kafka:9092", "kafka_topic"="{topic}",
           "property.kafka_default_offsets"="OFFSET_BEGINNING")'''
print(load_sql)
lab.execute(load_sql)

## 2. 写入两笔订单，等待查询可见

发送成功不等于 Doris 查询已经可见。轮询有明确超时；若超时，查看 `SHOW ROUTINE LOAD` 的暂停原因与错误行，不要跳过验收。

In [ ]:
rows = [dict(order_id=910001, customer_id=1, order_status="CREATED", order_amount="100.00"),
        dict(order_id=910002, customer_id=2, order_status="CREATED", order_amount="200.00")]
produce(topic, rows)
baseline = [[910001,1,"CREATED","100.00"],[910002,2,"CREATED","200.00"]]
expect(wait_rows(lab,"ext_kafka_orders",baseline), baseline)
lab.sql(f"SHOW ROUTINE LOAD FOR {job_name}", title="消费状态与 Progress")

## 3. 暂停消费，制造积压

先暂停并记录已提交进度，再发送第三笔订单。等待两个批次间隔后，表结果应不变。
暂停并不撤销已经提交的数据；这里先核对基线，避免将之前的在途数据误当成新数据。

In [ ]:
import time
lab.execute(f"PAUSE ROUTINE LOAD FOR {job_name}")
paused = lab.sql(f"SHOW ROUTINE LOAD FOR {job_name}")
expect(paused.iloc[0]["State"], "PAUSED")
produce(topic,[dict(order_id=910003,customer_id=3,order_status="CREATED",order_amount="50.00")])
time.sleep(12)
expect(lab.query("SELECT * FROM ext_kafka_orders ORDER BY order_id"), baseline)

## 4. 恢复消费，再投递重复和更新

Unique Key 将同一订单覆盖为当前状态，不保留所有业务历史。重复订单不增加逻辑行数，
但这不等于验证了任意故障下的端到端 exactly-once；乱序版本裁决见 Module 7。

In [ ]:
lab.execute(f"RESUME ROUTINE LOAD FOR {job_name}")
expected = baseline + [[910003,3,"CREATED","50.00"]]
expect(wait_rows(lab,"ext_kafka_orders",expected),expected)
produce(topic,[rows[0],dict(order_id=910001,customer_id=1,order_status="PAID",order_amount="100.00")])
expected[0][2] = "PAID"
expect(wait_rows(lab,"ext_kafka_orders",expected),expected)
lab.sql(f"SHOW ROUTINE LOAD FOR {job_name}", title="恢复后的进度")
expect(lab.query("SELECT COUNT(*),SUM(order_amount) FROM ext_kafka_orders"), [[3,"350.00"]])

## 5. 停止任务，保留结果

验收：暂停期间仍两行，恢复后最终三行 / 350.00，订单 910001 为 PAID；提交暂停前后进度截图。
STOP 为终态，不同于可恢复的 PAUSE。完成后删除本次专用 Topic，不影响其他 Topic；Doris 结果保留。

独立练习：另建一个 Topic / 目标表，将 `max_error_number=0` 与非法金额结合，查看任务状态及错误原因。不要往当前基线混入坏数据。

In [ ]:
lab.execute(f"STOP ROUTINE LOAD FOR {job_name}")
print(kafka("kafka-topics.sh", "--delete", "--topic", topic))
lab.close()